In [11]:
from collections import namedtuple, Counter, defaultdict
import random
import math
import functools
cache = functools.lru_cache(10**6)

In [12]:
class Game:
    """A game is similar to a problem, but it has a terminal test instead of
    a goal test, and a utility for each terminal state. To create a game,
    subclass this class and implement `actions`, `result`, `is_terminal`,
    and `utility`. You will also need to set the .initial attribute to the
    initial state; this can be done in the constructor."""

    def actions(self, state):
        """Return a collection of the allowable moves from this state."""
        raise NotImplementedError

    def result(self, state, move):
        """Return the state that results from making a move from a state."""
        raise NotImplementedError

    def is_terminal(self, state):
        """Return True if this is a final state for the game."""
        return not self.actions(state)

    def to_move(self, state):
        return state.to_move

    def terminal_test(self, state):
        return self.is_terminal(state)

    def utility(self, state, player):
        """Return the value of this final state to player."""
        raise NotImplementedError


def play_game(game, strategies: dict, verbose=False):
    """Play a turn-taking game. `strategies` is a {player_name: function} dict,
    where function(state, game) is used to get the player's move."""
    state = game.initial
    while not game.is_terminal(state):
        player = state.to_move
        move = strategies[player](game, state)
        state = game.result(state, move)
        if verbose:
            print('Player', player, 'move:', move)
            print(state)
    return state

In [18]:
class TicTacToe(Game):
    """Play TicTacToe on an `height` by `width` board, needing `k` in a row to win.
    'X' plays first against 'O'."""

    def __init__(self, height=6, width=6, k=4):
        self.k = k # k in a row
        self.squares = {(x, y) for x in range(width) for y in range(height)}
        self.initial = Board(height=height, width=width, to_move='X', utility=0)

    def actions(self, board):
        """Legal moves are any square not yet taken."""
        return self.squares - set(board)

    def result(self, board, square):
        """Place a marker for current player on square."""
        player = board.to_move
        board = board.new({square: player}, to_move=('O' if player == 'X' else 'X'))
        win = k_in_row(board, player, square, self.k)
        board.utility = (0 if not win else +1 if player == 'X' else -1)
        return board

    def utility(self, board, player):
        """Return the value to player; 1 for win, -1 for loss, 0 otherwise."""
        return board.utility if player == 'X' else -board.utility

    def is_terminal(self, board):
        """A board is a terminal state if it is won or there are no empty squares."""
        return board.utility != 0 or len(self.squares) == len(board)

    def display(self, board): print(board)


def k_in_row(board, player, square, k):
    """True if player has k pieces in a line through square."""
    def in_row(x, y, dx, dy): return 0 if board[x, y] != player else 1 + in_row(x + dx, y + dy, dx, dy)
    return any(in_row(*square, dx, dy) + in_row(*square, -dx, -dy) - 1 >= k
               for (dx, dy) in ((0, 1), (1, 0), (1, 1), (1, -1)))

def heuristic(game, board, player):
    """Estimate board value for player."""
    score = 0
    k = game.k

    # Count potential lines for both players
    for (dx, dy) in ((0,1),(1,0),(1,1),(1,-1)):
        for x in range(board.width):
            for y in range(board.height):
                line = []
                for i in range(k):
                    xx, yy = x + dx*i, y + dy*i
                    if 0 <= xx < board.width and 0 <= yy < board.height:
                        line.append(board[xx, yy])
                    else:
                        break
                if len(line) == k:
                    if all(c in ('.', player) for c in line):
                        score += 1
                    opp = 'O' if player == 'X' else 'X'
                    if all(c in ('.', opp) for c in line):
                        score -= 1
    return score

def move_order(game, board, move):
    """Prioritize moves near existing pieces."""
    x, y = move
    score = 0
    for dx in (-1,0,1):
        for dy in (-1,0,1):
            if board[x+dx, y+dy] != '.':
                score += 1
    return score


In [19]:
class Board(defaultdict):
    """A board has the player to move, a cached utility value,
    and a dict of {(x, y): player} entries, where player is 'X' or 'O'."""
    empty = '.'
    off = '#'

    def __init__(self, width=8, height=8, to_move=None, **kwds):
        self.__dict__.update(width=width, height=height, to_move=to_move, **kwds)

    def new(self, changes: dict, **kwds) -> 'Board':
        "Given a dict of {(x, y): contents} changes, return a new Board with the changes."
        board = Board(width=self.width, height=self.height, **kwds)
        board.update(self)
        board.update(changes)
        return board

    def __missing__(self, loc):
        x, y = loc
        if 0 <= x < self.width and 0 <= y < self.height:
            return self.empty
        else:
            return self.off

    def __hash__(self):
        return hash(tuple(sorted(self.items()))) + hash(self.to_move)

    def __repr__(self):
        def row(y): return ' '.join(self[x, y] for x in range(self.width))
        return '\n'.join(map(row, range(self.height))) +  '\n'


In [20]:
def alphabeta_search(game, state, max_depth=4):
    """Depth-limited alpha-beta with heuristic evaluation."""
    infinity = math.inf
    player = state.to_move

    @cache
    def max_value(state, alpha, beta, depth):
        if game.is_terminal(state):
            return game.utility(state, player), None
        if depth == 0:
            return heuristic(game, state, player), None

        v, best_move = -infinity, None
        moves = sorted(game.actions(state), key=lambda m: move_order(game, state, m), reverse=True)

        for a in moves:
            v2, _ = min_value(game.result(state, a), alpha, beta, depth-1)
            if v2 > v:
                v, best_move = v2, a
            if v >= beta:
                return v, best_move
            alpha = max(alpha, v)
        return v, best_move

    @cache
    def min_value(state, alpha, beta, depth):
        if game.is_terminal(state):
            return game.utility(state, player), None
        if depth == 0:
            return heuristic(game, state, player), None

        v, best_move = infinity, None
        moves = sorted(game.actions(state), key=lambda m: move_order(game, state, m))

        for a in moves:
            v2, _ = max_value(game.result(state, a), alpha, beta, depth-1)
            if v2 < v:
                v, best_move = v2, a
            if v <= alpha:
                return v, best_move
            beta = min(beta, v)
        return v, best_move

    return max_value(state, -infinity, +infinity, max_depth)


In [23]:
def random_player(game, state): return random.choice(list(game.actions(state)))

def player(search_algorithm):
    """A game player who uses the specified search algorithm"""
    return lambda game, state: search_algorithm(game, state)[1]

def query_player(game, state):
    print("current state:")
    game.display(state)
    print("available moves: {}".format(game.actions(state)))
    print("")
    move = None

    legal_moves = game.actions(state)

    while True:
        move_string = input('Your move? ')
        try:
            move = eval(move_string)
        except Exception:
            print("Invalid format. Enter a tuple like (x, y).")
            continue

        if move not in legal_moves:
            print("Illegal move! That square is already taken or out of bounds.")
            continue

        return move

In [24]:
play_game(TicTacToe(), dict(X=query_player, O=player(alphabeta_search)), verbose=True).utility

current state:
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .

available moves: {(4, 0), (3, 4), (4, 3), (3, 1), (5, 4), (5, 1), (0, 2), (0, 5), (2, 2), (1, 0), (2, 5), (1, 3), (4, 2), (3, 0), (4, 5), (3, 3), (5, 0), (5, 3), (0, 1), (2, 4), (1, 2), (0, 4), (2, 1), (1, 5), (3, 2), (4, 1), (3, 5), (5, 2), (4, 4), (5, 5), (0, 0), (1, 1), (0, 3), (2, 0), (1, 4), (2, 3)}

Player X move: (4, 0)
. . . . X .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .

Player O move: (2, 2)
. . . . X .
. . . . . .
. . O . . .
. . . . . .
. . . . . .
. . . . . .

current state:
. . . . X .
. . . . . .
. . O . . .
. . . . . .
. . . . . .
. . . . . .

available moves: {(3, 4), (4, 3), (3, 1), (5, 4), (5, 1), (0, 2), (0, 5), (1, 0), (2, 5), (1, 3), (4, 2), (3, 0), (4, 5), (3, 3), (5, 0), (5, 3), (0, 1), (2, 4), (1, 2), (0, 4), (2, 1), (1, 5), (3, 2), (4, 1), (3, 5), (5, 2), (4, 4), (5, 5), (0, 0), (1, 1), (0, 3), (2, 0), (1, 4), (2, 3)}

Illegal move! That square is already

KeyboardInterrupt: Interrupted by user